# Preparacion de Datos - Speed Dating Colombia

Preparacion completa de datos para analisis de citas rapidas.

In [ ]:
import matplotlib
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)
PALETA_ROSA = ["#FF1493", "#FF69B4", "#FFB6C1", "#FFC0CB"]
sns.set_palette(PALETA_ROSA)
pd.set_option("display.max_columns", None)
print("Librerias importadas")

## 1. Carga de Datos

In [ ]:
df = pd.read_csv("../data/Speed Dating Data.csv", encoding="latin-1")
print("Dimensiones: %s" % str(df.shape))
df.head()

## 2. Diccionario

In [ ]:
dicc = {"iid":"ID participante","gender":"Genero","age":"Edad","attr":"Atractivo","sinc":"Sinceridad","intel":"Inteligencia","fun":"Diversion","amb":"Ambicion","shar":"Intereses","like":"Gusto","prob":"Probabilidad","age_o":"Edad companero","race_o":"Raza companero","samerace":"Misma raza","met":"Conocidos","match":"Match TARGET"}
df_dict = pd.DataFrame(list(dicc.items()), columns=["Variable","Descripcion"])
print("Variables: %d" % len(dicc))
df_dict

## 3. Pandas Profiling

In [ ]:
import os
os.makedirs("../reports", exist_ok=True)
try:
    from ydata_profiling import ProfileReport
except:
    from pandas_profiling import ProfileReport
profile = ProfileReport(df, title="Speed Dating", explorative=True, minimal=True)
profile.to_file("../reports/pandas_profiling.html")
print("Guardado: reports/pandas_profiling.html")

## 4. Distribucion Target

In [ ]:
mc = df["match"].value_counts()
print("No Match: %d (%.1f%%)" % (mc[0], mc[0]*100.0/len(df)))
print("Match: %d (%.1f%%)" % (mc[1], mc[1]*100.0/len(df)))
fig, ax = plt.subplots(1,2,figsize=(15,6))
bars = ax[0].bar(["No Match","Match"],[mc[0],mc[1]],color=["#FFB6C1","#FF1493"],edgecolor="black")
ax[0].set_title("Distribucion",fontweight="bold")
ax[0].set_ylabel("Frecuencia")
for b in bars:
    h=b.get_height()
    ax[0].text(b.get_x()+b.get_width()/2.,h,"%.0f" % h,ha="center",va="bottom",fontweight="bold")
ax[1].pie([mc[0],mc[1]],labels=["No Match","Match"],autopct="%%1.1f%%",colors=["#FFB6C1","#FF1493"])
ax[1].set_title("Proporcion",fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/distribucion_target.png",dpi=300,bbox_inches="tight")
plt.show()

## 5. Estadistica

In [ ]:
nc = df.select_dtypes(include=[np.number]).columns.tolist()
print("Variables numericas: %d" % len(nc))
desc = df[nc].describe().T
desc["missing"] = df[nc].isnull().sum()
desc["pct_miss"] = (df[nc].isnull().sum()/len(df)*100).round(2)
print(desc.head(10).to_string())
desc.to_csv("../reports/estadisticas_descriptivas.csv")
plt.figure(figsize=(16,14))
cm = df[nc].corr()
mask = np.triu(np.ones_like(cm,dtype=bool))
sns.heatmap(cm,mask=mask,cmap="RdBu_r",center=0,square=True,linewidths=0.5,cbar_kws={"shrink":.8},fmt=".2f",annot=False,vmin=-1,vmax=1)
plt.title("Matriz Correlacion",fontweight="bold",pad=20)
plt.tight_layout()
plt.savefig("../reports/matriz_correlacion_inicial.png",dpi=300,bbox_inches="tight")
plt.show()

## 6. Limpieza de Nulos

In [ ]:
dfc = df.copy()
dfc.drop(columns=["iid","id","idg","partner","pid","wave","round","position","positin1","order","condtn","undergrd","zipcode","career","from","field"],inplace=True,errors="ignore")
print("Columnas: %d" % dfc.shape[1])

## ELIMINACION DE VARIABLES CON DATA LEAKAGE

In [ ]:
# Estas variables causan que el modelo haga trampa (ROC-AUC = 1.0)
# porque son componentes directas del target 'match' o se miden post-evento
LEAKAGE_VARS = ['dec', 'dec_o', 'like', 'prob', 'match_es']
leakage_encontradas = [c for c in LEAKAGE_VARS if c in dfc.columns]
dfc.drop(columns=leakage_encontradas, inplace=True, errors='ignore')
print('Variables de leakage eliminadas:', leakage_encontradas)
print('Columnas restantes:', dfc.shape[1])

## 7. Nulos > 50%

In [ ]:
total_nulos = dfc.isnull().sum().sum()
print("Nulos totales: %d" % total_nulos)
col50 = dfc.isnull().sum()[dfc.isnull().sum()/len(dfc)>0.5].index.tolist()
if col50:
    dfc.drop(columns=col50,inplace=True)
    print("Eliminadas >50%% nulos: %s" % col50)
dfc.to_csv("../data/data_prepared.csv", index=False)
print("Guardado: data/data_prepared.csv")

## 8. Imputacion

In [ ]:
numc = dfc.select_dtypes(include=[np.number]).columns.tolist()
if "match" in numc:
    numc.remove("match")
for col in numc:
    if dfc[col].isnull().sum()>0:
        med=dfc[col].median()
        dfc[col].fillna(med,inplace=True)
        print("%s: mediana=%.2f" % (col,med))
print("Nulos restantes: %d" % dfc.isnull().sum().sum())

## 9. Winsorizing

In [ ]:
scols = [c for c in dfc.select_dtypes(include=[np.number]).columns if c not in ["match","gender","race","race_o","samerace","goal","date","go_out","field_cd","career_c","met","dec","dec_o","int_corr"]]
for col in scols:
    if col in dfc.columns:
        p5=dfc[col].quantile(0.05)
        p95=dfc[col].quantile(0.95)
        dfc[col]=dfc[col].clip(lower=p5,upper=p95)
print("Winsorizing completado")

## 10. Redundancia

In [ ]:
corr_m = dfc.corr()
tc = corr_m["match"].abs().sort_values(ascending=False)
print(tc.head(15))
irr = tc[tc<0.02].index.tolist()
irr = [c for c in irr if c!="match"]
print("Irrelevantes: %d" % len(irr))
red=[]
ca=corr_m.abs()
np.fill_diagonal(ca.values,0)
for i in range(len(ca.columns)):
    for j in range(i+1,len(ca.columns)):
        if ca.iloc[i,j]>0.85:
            red.append((ca.columns[i],ca.columns[j],ca.iloc[i,j]))
print("Redundantes: %d" % len(red))
for r in red[:10]:
    print("  %s - %s: %.3f" % (r[0],r[1],r[2]))